# CoNLL-2003 NER — reproducible experiment

**Andrea Bianco | ENSAI | BiLSTM vs DistilBERT**

This Google Colab notebook delegates modeling to the tested Python modules in the repository, rather than duplicating them. Requires a GPU runtime for efficient DistilBERT training, a ZIP of this repository, and an authorized copy of the original `Project.zip`. **No dataset is committed.** Saved October 2026 results are already available in `results/`; executing this notebook is optional. See `METHODOLOGY_AUDIT.md` for limitations of the subword-expanded entity F1.

## 1. Load the two ZIP files

Upload the current repository ZIP and the original ENSAI `Project.zip` together. Files remain in the temporary Colab session.

In [ ]:
import os, sys, zipfile, shutil, subprocess
from pathlib import Path
from google.colab import files
import torch
print("CUDA available:", torch.cuda.is_available())
uploaded = files.upload()
archives = [Path(p) for p in uploaded if p.lower().endswith(".zip")]
if len(archives) != 2:
    raise ValueError("Exactly two ZIP files are required")
repo_zip = data_zip = None
for p in archives:
    with zipfile.ZipFile(p) as z:
        names = z.namelist()
        if any(x.endswith("/run_experiment.py") for x in names):
            repo_zip = p
        if any(x.endswith("/conll03-iob-pos.json.gz") for x in names):
            data_zip = p
if not repo_zip or not data_zip or repo_zip == data_zip:
    raise ValueError("Could not identify repository and original data archives")
workspace = Path("/content/ner_reproduction")
with zipfile.ZipFile(repo_zip) as z:
    for entry in z.infolist():
        if entry.is_dir():
            continue
        rel = Path(entry.filename)
        if rel.is_absolute() or ".." in rel.parts:
            raise ValueError("Unsafe archive member")
        target = workspace / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(entry) as src, target.open("wb") as dst:
            shutil.copyfileobj(src, dst)
runners = list(workspace.rglob("run_experiment.py"))
if len(runners) != 1:
    raise ValueError("Expected one run_experiment.py")
repo = runners[0].parent
with zipfile.ZipFile(data_zip) as z:
    members = [x for x in z.namelist() if x.endswith("/conll03-iob-pos.json.gz")]
    if len(members) != 1:
        raise ValueError("Expected one dataset")
    dataset = repo / "data/raw/conll03-iob-pos.json.gz"
    dataset.parent.mkdir(parents=True, exist_ok=True)
    with z.open(members[0]) as src, dataset.open("wb") as dst:
        shutil.copyfileobj(src, dst)
os.chdir(repo)
print("Ready:", repo, dataset.relative_to(repo))

## 2. Install dependencies and run tests

The setuptools workaround below is the procedure that succeeded on the original October 2026 Colab runtime. It is not guaranteed for future runtime versions. Failures stop the workflow before training.

In [ ]:
def run(*args):
    subprocess.run([str(a) for a in args], check=True)
run(sys.executable, "-m", "pip", "install", "-q", "setuptools==69.5.1", "wheel")
run(sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5", "scikit-learn>=1.3,<2")
run(sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "seqeval==1.2.2")
from seqeval.metrics import f1_score
run(sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v")
run(sys.executable, "-m", "src.validate_data", dataset, "--original-split-counts")
print("Dependency import, tests and dataset validation: OK")

## 3. Run both full-dataset experiments

BiLSTM: 10 epochs, AdamW 5e-4. DistilBERT: 3 epochs, AdamW 1e-5. Both use seed 42 and batch size 32. Results are saved under ignored `outputs/`; this cell **does not overwrite the recorded evidence in `results/`**.

In [ ]:
import json
for model in ("bilstm", "distilbert"):
    run(sys.executable, "run_experiment.py", "--model", model, "--data", dataset)
    run(sys.executable, "compare_results.py", f"outputs/{model}.json")
    result = json.loads(Path(f"outputs/{model}.json").read_text(encoding="utf-8"))
    print(model, "test entity F1:", round(result["test_entities"]["f1"], 4))
    files.download(f"outputs/{model}.json")

## Interpretation

Saved original ENSAI notebook entity F1: **74.19% BiLSTM**, **86.16% DistilBERT**. October 2026 full-data Colab run: **69.94% BiLSTM**, **85.78% DistilBERT**. They are not identical numerical reproductions. Metrics follow the original **subword-expanded BIO** evaluation; they should not be described as canonical word-level CoNLL-2003 scores. See `METHODOLOGY_AUDIT.md`. No outputs are embedded in this notebook.